# Discover zoonotic bacterial species

This notebook searches PubMed for zoonosis-related articles and asks an LLM to extract bacterial species explicitly associated with zoonosis. It uses two levels: a title-only pass for candidate discovery, followed by an abstract pass for evidence and species names requiring context.

PubMed records, LLM decisions, and the final one-row-per-species dataframe are persisted locally. The last cell reconstructs the result from files, so it remains usable after manually interrupting either LLM stage.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    DEFAULT_ZOONOSIS_QUERY,
    OpenAIChatCompleter,
    PubMedClient,
    build_bacterial_species_summary,
    collect_zoonosis_articles,
    run_species_extraction_stage,
)

## Configure PubMed, the LLM, and checkpoints

In [ ]:
from graphicalizer.notebook_config import configured_env, load_notebook_config, resolve_config_path

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_01_zoonotic_bacterial_species']

PUBMED_EMAIL = COMMON['pubmed_email']
PUBMED_API_KEY = configured_env(COMMON, 'pubmed_api_key_env')
PUBMED_MAX_RESULTS = SETTINGS['pubmed_max_results']
PUBMED_SEARCH_PAGE_SIZE = SETTINGS['pubmed_search_page_size']
PUBMED_FETCH_BATCH_SIZE = SETTINGS['pubmed_fetch_batch_size']
QUERY = SETTINGS['query']

OUTPUT_DIR = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root']) / SETTINGS['output_subdir'],
)
ARTICLES_PATH = OUTPUT_DIR / SETTINGS['articles_filename']
TITLE_EXTRACTIONS_PATH = OUTPUT_DIR / SETTINGS['title_extractions_filename']
ABSTRACT_EXTRACTIONS_PATH = OUTPUT_DIR / SETTINGS['abstract_extractions_filename']
SUMMARY_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['summary_path'])

OPENAI_MODEL = configured_env(COMMON, 'openai_model_env', COMMON['openai_model_default'])
OPENAI_API_KEY = configured_env(COMMON, 'openai_api_key_env')
LLM_MAX_TOKENS = SETTINGS['openai_max_tokens']
LLM_RETRIES = COMMON['llm_retries']
LLM_MAX_CALLS = COMMON['llm_max_calls']
LLM_SAVE_EVERY = COMMON['llm_save_every']
RESUME = COMMON['resume']
RETRY_FAILED_LLM_ROWS = COMMON['retry_failed_llm_rows']
RUN_ABSTRACT_STAGE = SETTINGS['run_abstract_stage']
ABSTRACT_INPUT_MODE = SETTINGS['abstract_input_mode']

pubmed = PubMedClient(
    email=PUBMED_EMAIL,
    api_key=PUBMED_API_KEY,
    tool=COMMON['pubmed_tool'],
    retries=COMMON['pubmed_retries'],
)


## Preview the zoonosis query

In [ ]:
print(QUERY)

## Acquire and persist the PubMed article corpus

The search manifest and article table are written incrementally. A rerun fetches only records that were not completed successfully.

In [ ]:
articles = collect_zoonosis_articles(
    pubmed,
    OUTPUT_DIR,
    query=QUERY,
    max_results=PUBMED_MAX_RESULTS,
    search_page_size=PUBMED_SEARCH_PAGE_SIZE,
    fetch_batch_size=PUBMED_FETCH_BATCH_SIZE,
    resume=RESUME,
)
print('PubMed article rows:', len(articles))
display(articles[['pmid', 'title', 'publication_date', 'fetch_status']].head(10))

## Level 1: extract bacterial species from titles

Only titles are supplied to the LLM in this stage. The decision table is checkpointed after every `LLM_SAVE_EVERY` calls and also flushed when the cell is interrupted.

In [ ]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the LLM extraction stages.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
title_inputs = articles[articles['title'].fillna('').str.strip().ne('')].copy()
title_extractions = run_species_extraction_stage(
    title_inputs,
    llm,
    TITLE_EXTRACTIONS_PATH,
    stage='title',
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Title decisions:', len(title_extractions))
display(title_extractions[['pmid', 'title', 'associated_with_zoonosis', 'bacterial_species', 'confidence']].head(20))

## Select records for the abstract stage

The default is a precision-first funnel: abstract review is run on articles whose title pass found at least one bacterial species. Set `ABSTRACT_INPUT_MODE = 'all_articles'` above when recall is more important than LLM cost.

In [ ]:
def _has_species(value):
    try:
        return bool(json.loads(value))
    except (TypeError, ValueError, json.JSONDecodeError):
        return False

if ABSTRACT_INPUT_MODE == 'all_articles':
    abstract_pmids = set(articles['pmid'].astype(str))
elif ABSTRACT_INPUT_MODE == 'title_positive':
    abstract_pmids = set(
        title_extractions.loc[
            title_extractions['associated_with_zoonosis'].fillna(False)
            & title_extractions['bacterial_species'].map(_has_species),
            'pmid',
        ].astype(str)
    )
else:
    raise ValueError("ABSTRACT_INPUT_MODE must be 'title_positive' or 'all_articles'.")

abstract_inputs = articles[
    articles['pmid'].astype(str).isin(abstract_pmids)
    & articles['abstract'].fillna('').str.strip().ne('')
].copy()
print('Abstract-stage input rows:', len(abstract_inputs))
display(abstract_inputs[['pmid', 'title']].head(20))

## Level 2: extract bacterial species from abstracts

In [ ]:
if RUN_ABSTRACT_STAGE:
    abstract_extractions = run_species_extraction_stage(
        abstract_inputs,
        llm,
        ABSTRACT_EXTRACTIONS_PATH,
        stage='abstract',
        model=OPENAI_MODEL,
        max_tokens=LLM_MAX_TOKENS,
        retries=LLM_RETRIES,
        max_llm_calls=LLM_MAX_CALLS,
        save_every=LLM_SAVE_EVERY,
        resume=RESUME,
        retry_failed=RETRY_FAILED_LLM_ROWS,
    )
    print('Abstract decisions:', len(abstract_extractions))
    display(abstract_extractions[['pmid', 'title', 'associated_with_zoonosis', 'bacterial_species', 'confidence']].head(20))
else:
    print('Abstract stage disabled; the recovery cell can still build a title-only summary.')

## Persist the species dataframe

In [ ]:
species_summary = build_bacterial_species_summary(
    title_extractions,
    abstract_extractions if RUN_ABSTRACT_STAGE else pd.DataFrame(),
    SUMMARY_PATH,
)
print('Bacterial species:', len(species_summary))
display(species_summary)

## Rebuild from persisted files after an interrupt

Run this cell independently after stopping a title or abstract extraction cell. It does not use the article or extraction variables from memory.

In [ ]:
def _load_checkpoint(path):
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

persisted_title_extractions = _load_checkpoint(TITLE_EXTRACTIONS_PATH)
persisted_abstract_extractions = _load_checkpoint(ABSTRACT_EXTRACTIONS_PATH)
if persisted_title_extractions.empty and persisted_abstract_extractions.empty:
    raise FileNotFoundError(
        'No extraction checkpoint found. Run the title stage first, then rerun this cell.'
    )
species_summary = build_bacterial_species_summary(
    persisted_title_extractions,
    persisted_abstract_extractions,
    SUMMARY_PATH,
)
print('Rebuilt from checkpoints:')
print('  title rows:', len(persisted_title_extractions))
print('  abstract rows:', len(persisted_abstract_extractions))
print('  species rows:', len(species_summary))
print('  saved to:', SUMMARY_PATH)
display(species_summary)

In [ ]:
print('Checkpoint directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
print('Species dataframe exists:', SUMMARY_PATH.exists())